# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ammad-Nasir/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
'''
## 1) My lane as an ML task

My lane is **Content Refresh / Page Prioritization**.

I see this mainly as a **ranking/scoring problem**. The goal is not just to say whether a page is good or bad. I want to give pages a priority score and use that score to decide which pages should be checked first.

The unit I am working with is one webpage. The output would be a ranked list of pages, with the pages that look more likely to need attention near the top.

This connects to the action from Week 1 because a content team could use the ranked list to decide which pages to review first.
'''

'\n## 1) My lane as an ML task\n\nMy lane is **Content Refresh / Page Prioritization**.\n\nI see this mainly as a **ranking/scoring problem**. The goal is not just to say whether a page is good or bad. I want to give pages a priority score and use that score to decide which pages should be checked first.\n\nThe unit I am working with is one webpage. The output would be a ranked list of pages, with the pages that look more likely to need attention near the top.\n\nThis connects to the action from Week 1 because a content team could use the ranked list to decide which pages to review first.\n'

In [6]:
import pandas as pd

# The dataset is hosted online, so we use the URL directly.
data_path = "https://raw.githubusercontent.com/Ammad-Nasir/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

# Load dataset
df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (30000, 44)


In [7]:
display(df.head())

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create the target/proxy
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print(df["is_declining_label"].value_counts())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [9]:
target_summary = pd.DataFrame({
    "target_value": [0, 1],
    "meaning": [
        "Not showing a downward trend",
        "Showing a downward trend"
    ],
    "count": [
        (df["is_declining_label"] == 0).sum(),
        (df["is_declining_label"] == 1).sum()
    ]
})

display(target_summary)

,target_value,meaning,count
0,0,Not showing a downward trend,13738
1,1,Showing a downward trend,16262




For the target, I will use whether a page is showing a downward trend as a proxy for pages that may need attention.

The starter pipeline defines this as:

`is_declining_label = (trend_direction == "down")`

So the target will be:

* `1` if the page has a downward trend
* `0` otherwise

I am calling this a proxy because a downward trend does not automatically mean that the page needs a content refresh. It is just a useful signal for deciding which pages may be worth reviewing.

I would not use `trend_direction` or `trend_pct` as model features because they are used to create the target. Using them as features would leak the answer into the model.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
Precision@50



My main success metric will be **Precision@50**.

I chose this because the main goal is to find a small number of pages that are worth reviewing first. I do not need the model to classify every page perfectly.

Precision@50 tells me how many of the top 50 pages selected by the model are actually in the target group.

For example, if 30 out of the top 50 pages are actually declining, the Precision@50 would be:

30 / 50 = 0.60

This metric fits the business action because a content team may only have enough time to review a limited number of pages at a time.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
unit_of_analysis = df[
    [
        "trend_direction",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "is_declining_label"
    ]
].head(10)

display(unit_of_analysis)

,trend_direction,impressions_90d,clicks_90d,ctr,is_declining_label
0,down,3803,29,0.76,1
1,down,15320,7,0.05,1
2,down,12581,11,0.09,1
3,stable,11751,58,0.49,0
4,down,19140,24,0.13,1
5,down,3970,1,0.03,1
6,down,20,0,0.00,1
7,stable,1724,1,0.06,0
8,down,32574,29,0.09,1
9,down,1240,2,0.16,1


In [11]:
declining_count = (df["is_declining_label"] == 1).sum()
total_pages = len(df)

print("Total pages:", total_pages)
print("Declining pages:", declining_count)
print(
    "Percentage declining:",
    round((declining_count / total_pages) * 100, 2),
    "%"
)

Total pages: 30000
Declining pages: 16262
Percentage declining: 54.21 %


### What one row represents

One row represents **one webpage**.

The columns contain different information about that page, such as its search-performance measurements and trend information.

For this task, I want to use the information available for each page to estimate whether it should receive a higher priority for review.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
signals = [
    "impressions_90d",
    "clicks_90d",
    "ctr"
]

display(df[signals].describe())

,impressions_90d,clicks_90d,ctr
count,30000.000000,30000.000000,30000.000000
mean,5200.366300,16.097333,0.510733
std,16838.019547,75.076958,3.279162
min,1.000000,0.000000,0.000000
25%,81.000000,0.000000,0.000000
50%,731.000000,1.000000,0.070000
75%,3615.250000,7.000000,0.290000
max,517715.000000,4178.000000,100.000000




A fixed rule could be something simple like saying that every page with low clicks or a certain amount of impressions should be reviewed.

The problem is that pages can have different combinations of search signals. One fixed threshold may work for some pages but not for others.

ML could look at several signals together and learn patterns from the examples in the data. It can then give each page a score instead of using only one fixed condition.

I would still compare the ML approach against a simple rule. If the ML model cannot improve the ranking enough to be useful, then there would not be a strong reason to use ML.

The goal is therefore not to use ML just because it is more complicated. The goal is to see whether it can produce a better prioritized list than a simple rule.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.